In [1]:
import os
import sys

# Go 3 levels up from components → project root
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
sys.path.append(project_root)

print("Added to path:", project_root)

Added to path: c:\Users\Gunjan\AI_Project


In [2]:
import os
import sys
import pickle
from dataclasses import dataclass
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from src.IDS.exception import CustomException
from src.IDS.logger import logging
from dataclasses import dataclass

In [3]:
import src
print("SRC is now recognized successfully")

SRC is now recognized successfully


In [4]:
from src.IDS.utils import read_sql_data

monday_df, tue_fri_df = read_sql_data()

print("Monday shape:", monday_df.shape)
print("Tue-Fri shape:", tue_fri_df.shape)

c:\Users\Gunjan\AI_Project\src\IDS\utils.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tue_fri_df = pd.read_sql_query("SELECT * FROM tuetofri_ids", mydb)
c:\Users\Gunjan\AI_Project\src\IDS\utils.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  monday_df = pd.read_sql_query("SELECT * FROM mon_ids", mydb)


Monday shape: (458831, 44)
Tue-Fri shape: (1621085, 44)


In [5]:
print(tue_fri_df["Attack"].value_counts())

Attack
0    1291600
1     329485
Name: count, dtype: int64


In [6]:
monday_df = monday_df.dropna()

if "Attack" in monday_df.columns:
    monday_df = monday_df.drop("Attack", axis=1)


In [7]:
scaler = StandardScaler()
X_monday_scaled = scaler.fit_transform(monday_df)



In [8]:
iso_model = IsolationForest(
    n_estimators=300,
    contamination=0.01,
    random_state=42
)

In [9]:
iso_model.fit(X_monday_scaled)

print("Isolation Forest trained successfully")

Isolation Forest trained successfully


In [10]:
tue_fri_df = tue_fri_df.dropna()

In [11]:
y = tue_fri_df["Attack"].astype(int)
X = tue_fri_df.drop("Attack", axis=1)

In [12]:
X_scaled = scaler.transform(X)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [14]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (1296868, 43)
Test shape: (324217, 43)


In [15]:
rf_model = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42
)

In [16]:
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

In [17]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9988680420829259
F1 Score: 0.9972141886609128

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    258320
           1       1.00      1.00      1.00     65897

    accuracy                           1.00    324217
   macro avg       1.00      1.00      1.00    324217
weighted avg       1.00      1.00      1.00    324217


Confusion Matrix:

[[258164    156]
 [   211  65686]]


In [24]:
print("Duplicates:", tue_fri_df.duplicated().sum())

Duplicates: 218154


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
import time

In [19]:
start = time.time()

log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("Logistic F1:", f1_score(y_test, y_pred_log))
print("Time:", time.time() - start)

Logistic F1: 0.9166090047958769
Time: 27.522801637649536


In [20]:
start = time.time()

et_model = ExtraTreesClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

et_model.fit(X_train, y_train)
y_pred_et = et_model.predict(X_test)

print("ExtraTrees F1:", f1_score(y_test, y_pred_et))
print("Time:", time.time() - start)

ExtraTrees F1: 0.9959679562625764
Time: 349.89798402786255


In [21]:
import lightgbm as lgb

start = time.time()

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31
)

lgb_model.fit(X_train, y_train)

y_pred_lgb = lgb_model.predict(X_test)

print("LightGBM F1:", f1_score(y_test, y_pred_lgb))
print("Time:", time.time() - start)
print(classification_report(y_test, y_pred_lgb))
print(confusion_matrix(y_test, y_pred_lgb))

[LightGBM] [Info] Number of positive: 263588, number of negative: 1033280
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.141396 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6190
[LightGBM] [Info] Number of data points in the train set: 1296868, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203250 -> initscore=-1.366106
[LightGBM] [Info] Start training from score -1.366106


c:\Users\Gunjan\AI_Project\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM F1: 0.9977608258377927
Time: 51.408727407455444
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    258320
           1       1.00      1.00      1.00     65897

    accuracy                           1.00    324217
   macro avg       1.00      1.00      1.00    324217
weighted avg       1.00      1.00      1.00    324217

[[258197    123]
 [   172  65725]]


In [22]:
import xgboost as xgb

start = time.time()

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    tree_method="hist"
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost F1:", f1_score(y_test, y_pred_xgb))
print("Time:", time.time() - start)
print(classification_report(y_test, y_pred_lgb))
print(confusion_matrix(y_test, y_pred_lgb))

XGBoost F1: 0.9974798463617179
Time: 41.83304977416992
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    258320
           1       1.00      1.00      1.00     65897

    accuracy                           1.00    324217
   macro avg       1.00      1.00      1.00    324217
weighted avg       1.00      1.00      1.00    324217

[[258197    123]
 [   172  65725]]


In [23]:
start = time.time()

mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    max_iter=50
)

mlp_model.fit(X_train, y_train)

y_pred_mlp = mlp_model.predict(X_test)

print("MLP F1:", f1_score(y_test, y_pred_mlp))
print("Time:", time.time() - start)

c:\Users\Gunjan\AI_Project\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


MLP F1: 0.984291190671125
Time: 4780.601125717163


In [25]:
def hybrid_predict(sample):
    sample_scaled = scaler.transform(sample)

    iso_pred = iso_model.predict(sample_scaled)

    if iso_pred[0] == 1:
        return "BENIGN"
    else:
        return rf_model.predict(sample_scaled)[0]